# FPL Features

## Building features with histroical gameweek data from multiple seasons

In [31]:
import pandas as pd

In [32]:
from pathlib import Path

RAW_PATH = Path("../data/raw/merged_gw_2022_2026.parquet")
seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]

if RAW_PATH.exists():
    all_df = pd.read_parquet(RAW_PATH)
    print("Loaded from local file")
else:
    frames = []
    for season in seasons:
        url = f"https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{season}/gws/merged_gw.csv"
        season_df = pd.read_csv(url)
        season_df["season"] = season
        frames.append(season_df)
    all_df = pd.concat(frames, ignore_index=True)
    all_df.to_parquet(RAW_PATH)
    print("Downloaded and saved")

Loaded from local file


In [33]:
all_df["kickoff_time"] = pd.to_datetime(all_df["kickoff_time"])
all_df = all_df.sort_values(["season", "element", "kickoff_time"]).reset_index(drop=True)

In [34]:
group = all_df.groupby(["season", "element"])["minutes"]

all_df["minutes_avg_last_5"] = group.transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

all_df["minutes_avg_last_3"] = group.transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

In [35]:
check = all_df[(all_df["season"] == "2025-26") & (all_df["name"].str.contains("Saka"))]
check[["GW", "kickoff_time", "minutes", "minutes_avg_last_3", "minutes_avg_last_5"]].head(10)

,GW,kickoff_time,minutes,minutes_avg_last_3,minutes_avg_last_5
84405,1,2025-08-17 15:30:00+00:00,90,NaN,NaN
84406,2,2025-08-23 16:30:00+00:00,52,90.000000,90.000000
84407,3,2025-08-31 15:30:00+00:00,0,71.000000,71.000000
84408,4,2025-09-13 11:30:00+00:00,0,47.333333,47.333333
84409,5,2025-09-21 15:30:00+00:00,45,17.333333,35.500000
84410,6,2025-09-28 15:30:00+00:00,69,15.000000,37.400000
84411,7,2025-10-04 14:00:00+00:00,90,38.000000,33.200000
84412,8,2025-10-18 16:30:00+00:00,90,68.000000,40.800000
84413,9,2025-10-26 14:00:00+00:00,65,83.000000,58.800000
84414,10,2025-11-01 15:00:00+00:00,90,81.666667,71.800000


In [36]:
ROLLING_COLS = ["minutes", "total_points", "expected_goals", "expected_assists", "bonus"]
all_df[ROLLING_COLS].dtypes

minutes               int64
total_points          int64
expected_goals      float64
expected_assists    float64
bonus                 int64
dtype: object

In [37]:
def add_rolling_features(df, cols, windows, group_cols=["season", "element"]):
    """For each column and window size, add the player's average over their previous matches."""
    df = df.copy()
    grouped = df.groupby(group_cols)

    for col in cols:
        for w in windows:
            new_name = f"{col}_avg_last_{w}"
            df[new_name] = grouped[col].transform(
                lambda s: s.shift(1).rolling(w, min_periods=1).mean()
            )
    return df

In [38]:
WINDOWS = [3, 5]
features_df = add_rolling_features(all_df, ROLLING_COLS, WINDOWS)

new_cols = [c for c in features_df.columns if "_avg_last_" in c]
print(len(new_cols))
new_cols

10


['minutes_avg_last_5',
 'minutes_avg_last_3',
 'total_points_avg_last_3',
 'total_points_avg_last_5',
 'expected_goals_avg_last_3',
 'expected_goals_avg_last_5',
 'expected_assists_avg_last_3',
 'expected_assists_avg_last_5',
 'bonus_avg_last_3',
 'bonus_avg_last_5']

In [39]:
saka = features_df[(features_df["season"] == "2025-26") & (features_df["name"].str.contains("Saka"))]
saka[["GW", "total_points", "total_points_avg_last_3"]].head(6)

,GW,total_points,total_points_avg_last_3
84405,1,3,NaN
84406,2,6,3.000000
84407,3,0,4.500000
84408,4,0,3.000000
84409,5,1,2.000000
84410,6,2,0.333333


In [40]:
all_df.groupby("season")["team"].nunique()

season
2022-23    20
2023-24    20
2024-25    20
2025-26    20
Name: team, dtype: int64

In [41]:
import numpy as np

team_matches = (
    all_df.drop_duplicates(subset=["season", "fixture", "team"])
    [["season", "fixture", "team", "was_home", "kickoff_time", "team_h_score", "team_a_score"]]
    .copy()
)

team_matches = team_matches.sort_values(["season", "team", "kickoff_time"]).reset_index(drop=True)

team_matches["goals_conceded"] = np.where(
    team_matches["was_home"],
    team_matches["team_a_score"],
    team_matches["team_h_score"]
)

team_matches["goals_scored"] = np.where(
    team_matches["was_home"],
    team_matches["team_h_score"],
    team_matches["team_a_score"]
)

In [42]:
team_matches = add_rolling_features(team_matches, cols=["goals_scored", "goals_conceded"], windows=WINDOWS, group_cols=["season", "team"])

arsenal = team_matches[(team_matches["season"] == "2025-26") & (team_matches["team"] == "Arsenal")]
arsenal[["kickoff_time", "goals_conceded", "goals_conceded_avg_last_3", "goals_conceded_avg_last_5", "goals_scored", "goals_scored_avg_last_3", "goals_scored_avg_last_5"]].head(7)

,kickoff_time,goals_conceded,goals_conceded_avg_last_3,goals_conceded_avg_last_5,goals_scored,goals_scored_avg_last_3,goals_scored_avg_last_5
2280,2025-08-17 15:30:00+00:00,0,NaN,NaN,1,NaN,NaN
2281,2025-08-23 16:30:00+00:00,0,0.000000,0.000000,5,1.000000,1.00
2282,2025-08-31 15:30:00+00:00,1,0.000000,0.000000,0,3.000000,3.00
2283,2025-09-13 11:30:00+00:00,0,0.333333,0.333333,3,2.000000,2.00
2284,2025-09-21 15:30:00+00:00,1,0.333333,0.250000,1,2.666667,2.25
2285,2025-09-28 15:30:00+00:00,1,0.666667,0.400000,2,1.333333,2.00
2286,2025-10-04 14:00:00+00:00,0,0.666667,0.600000,2,2.000000,2.20


In [43]:
opp_cols = ["goals_conceded_avg_last_3", "goals_conceded_avg_last_5",
            "goals_scored_avg_last_3", "goals_scored_avg_last_5"]

opp = team_matches[["season", "fixture", "was_home", "team"] + opp_cols].copy()
opp["was_home"] = ~opp["was_home"]

opp = opp.rename(columns={"team": "opp_team"}).rename(columns={c: f"opp_{c}" for c in opp_cols})

print("Rows before:", len(features_df))
features_df = features_df.merge(opp, on=["season", "fixture", "was_home"], how="left")
print("Rows after:", len(features_df))

Rows before: 113592
Rows after: 113592


In [44]:
saka = features_df[(features_df["season"] == "2025-26") & (features_df["name"].str.contains("Saka"))]
saka[["GW", "team", "opp_team", "was_home", "opp_goals_conceded_avg_last_5"]].head(5)

,GW,team,opp_team,was_home,opp_goals_conceded_avg_last_5
84405,1,Arsenal,Man Utd,False,NaN
84406,2,Arsenal,Leeds,True,0.000000
84407,3,Arsenal,Liverpool,False,2.000000
84408,4,Arsenal,Nott'm Forest,True,1.666667
84409,5,Arsenal,Man City,True,1.000000
